# Mini-lab — Skillbox / toolbox retrieval

Goal: see how an agent with many tools avoids context-window collapse by retrieving only the top-k relevant tools and skills for the current turn.


## 0. Why this pattern exists

If an agent has 50 tools, pasting every JSON schema into every prompt is wasteful and confusing. The toolbox stores tool definitions as vector-searchable rows. The skillbox does the same for prose playbooks: the prompt gets a compact manifest, and the full skill body is loaded only when needed.


In [ ]:
import json
from pathlib import Path

from agentic_memory_workshop.connection import load_env, connect
from agentic_memory_workshop.toolbox import upsert_tool, retrieve_tools, count_tools
from agentic_memory_workshop.skillbox import ingest_markdown_file, build_skill_manifest, get_skill, count_skills

load_env('../.env')
ROOT = Path('..').resolve()
connection = connect()
print('Connected to Oracle AI Database')


## 1. Ingest ten sample tools

Each tool has a name, description, and JSON parameters. `upsert_tool` stores the row and asks Oracle to compute `VECTOR_EMBEDDING(...)` in the database.


In [ ]:
tools = json.loads((ROOT / 'samples/tools.json').read_text())
try:
    for tool in tools:
        upsert_tool(connection, tool['name'], tool['description'], tool['parameters'], source='sample')
    print(f'Tools in registry: {count_tools(connection)}')
except Exception as exc:
    print('Tool ingest failed. Most common cause: the in-database ONNX model is not loaded yet.')
    print(type(exc).__name__, exc)
    raise


## 2. Retrieve only the relevant tools

Change the query and watch the top-k list shift. This is what goes into the model's active prompt instead of all tool schemas.


In [ ]:
query = 'The user wants to cancel a reservation and asks whether they can get a refund.'
for tool in retrieve_tools(connection, query, k=4):
    print(f"{tool['distance']:.3f}  {tool['name']} — {tool['description']}")


## 3. Ingest three prose skills

A skill is a markdown playbook. The agent sees only the short manifest until it calls `load_skill(name)`.


In [ ]:
for path in sorted((ROOT / 'samples/skills').glob('*.md')):
    info = ingest_markdown_file(connection, path)
    print('ingested', info)
print(f'Skills in registry: {count_skills(connection)}')


## 4. Build the prompt manifest

This compact manifest is safe to paste into the prompt. The full markdown body stays out of context until the agent explicitly loads it.


In [ ]:
manifest = build_skill_manifest(connection, query, k=2)
print(manifest)


## 5. Load one full skill body

This simulates the agent choosing a skill from the manifest and calling `load_skill(name)`.


In [ ]:
first_skill_name = manifest.split('\n')[1].split(':', 1)[0].removeprefix('- ').strip()
skill = get_skill(connection, first_skill_name)
print(skill['body'] if skill else 'No skill found')


## Checkpoint

Post a screenshot of your top-4 tools and top-2 skills. The important observation: the agent's prompt receives a small, query-shaped surface area rather than the whole registry.
